Q1. Embedding a query

In [1]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(q1)


In [2]:
v1[0]

np.float64(-0.020582036807885073)

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

Q2. Cosine similarity

In [4]:
doc = next(
    doc
    for doc in documents
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)
doc_content = doc["content"]
doc_embedding = embed.encode(doc_content)


In [5]:
doc_embedding.dot(v1)

np.float64(0.361070280302606)

Q3. Chunking and search by hand

In [6]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [7]:
chunk_contents = [chunk["content"] for chunk in chunks]
vectors = embed.encode_batch(chunk_contents)

len(vectors)


295

In [8]:
import numpy as np
X = np.array(vectors)

In [9]:
scores = X.dot(v1)

idx = np.argmax(scores)
idx, scores[idx]


(np.int64(94), np.float64(0.648901732433228))

In [10]:
chunks[94]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

Q4. Vector search with minsearch

In [11]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['content'])
vindex.fit(X, chunks)

In [12]:
q1 = "What metric do we use to evaluate a search engine?"

v1 = embed.encode(q1)

In [13]:
vindex.search(v1, num_results=5)

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

Q5. Text search vs vector search

In [14]:
# vector search with new query
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['content'])
vindex.fit(X, chunks)
q1 = "How do I store vectors in PostgreSQL?"
v1 = embed.encode(q1)
vindex.search(v1, num_results=5)

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [15]:
# indexsearch with new query

from minsearch import Index

index = Index(text_fields=['content'], keyword_fields=['content'])
index.fit(chunks)
index.search("How do I store vectors in PostgreSQL?", num_results=5)


[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

Q6. Hybrid search

In [16]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [17]:
# vector search with new query
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['content'])
vindex.fit(X, chunks)
q1 = "How do I give the model access to tools?"
v1 = embed.encode(q1)

vector_results = vindex.search(v1, num_results=5)

In [18]:
# indexsearch with new query

from minsearch import Index

index = Index(text_fields=['content'], keyword_fields=['content'])
index.fit(chunks)

text_results = index.search("How do I give the model access to tools?", num_results=5)


In [19]:
results = rrf([vector_results, text_results])
results

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 

# Homework Lesson 4

In [20]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [22]:
len(documents)

72

In [51]:
doc = documents[:3]
doc

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [25]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [23]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [24]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [33]:
import json
user_prompt = json.dumps(doc)

In [34]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [35]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [36]:
from evaluation_utils import llm_structured

In [42]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['What exactly is a RAG system, and how does it help an LLM answer questions better?', 'Why do LLMs make up answers or fail on recent info, and what are their main limits?', 'Why does this module build the search part and call the model directly in plain Python instead of using a framework?', 'What will be covered in the first part of the module before the agentic version starts?', 'What is the course project in this module, and what kind of FAQ problem is it trying to solve?']


In [44]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["content"]
    })

records

[{'question': 'What exactly is a RAG system, and how does it help an LLM answer questions better?',
  'document': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it su

In [45]:
from evaluation_utils import llm_structured_retry

In [47]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["content"]
        })

    return results, usage

**Q1. Generating questions**

In [48]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [50]:
import pandas as pd
pd.DataFrame(records)

,question,document
0,Why doesn't a plain LLM give a good answer whe...,# RAG\n\nVideo: [Watch this lesson](https://ww...
1,What changes when you put FAQ text into the pr...,# RAG\n\nVideo: [Watch this lesson](https://ww...
2,"What does RAG mean, and what are the two main ...",# RAG\n\nVideo: [Watch this lesson](https://ww...
3,How does the bot decide which FAQ entries to s...,# RAG\n\nVideo: [Watch this lesson](https://ww...
4,Why is the retrieval step so important in a RA...,# RAG\n\nVideo: [Watch this lesson](https://ww...


In [53]:
avg_input_tokens = sum(u.input_tokens for u in usages) / len(usages)
avg_input_tokens

1353.0

In [56]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [57]:
chunk_contents = [chunk["content"] for chunk in chunks]
vectors = embed.encode_batch(chunk_contents)

len(vectors)

295

In [62]:
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

def text_search(query: str, num_results: int = 5):
    return text_index.search(query, num_results=num_results)

def vector_search(query: str, num_results: int = 5):
    q_vec = embed.encode(query)
    return vector_index.search(q_vec, num_results=num_results)

In [63]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [64]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

**Q2. First result with text search**

In [65]:
q = ground_truth[0]["question"]

In [70]:
q

'What is a retrieval-augmented generation system, in simple terms, and why would you use it instead of relying on the model alone?'

In [68]:
text_results = text_search(q, num_results=5)
text_results[0]


{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

**Q3. First result with vector search**

In [69]:
vector_results = vector_search(q, num_results=5)
vector_results[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [83]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [84]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [85]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [86]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

In [90]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [97]:
doc_content_to_filename = {doc["content"]: doc["filename"] for doc in documents}

def compute_relevance(q, search_function):
    doc_filename = doc_content_to_filename[q["document"]]
    results = search_function(query=q["question"])

    return [int(d["filename"] == doc_filename) for d in results]


**Q4. Evaluating text search**

In [98]:

evaluate(ground_truth, text_search)

  0%|          | 0/15 [00:00<?, ?it/s]

{'hit_rate': 0.7333333333333333, 'mrr': 0.6222222222222221}

****Q5. Evaluating vector search****

In [99]:
evaluate(ground_truth, vector_search)

  0%|          | 0/15 [00:00<?, ?it/s]

{'hit_rate': 0.6666666666666666, 'mrr': 0.4488888888888889}

****Q6. Tuning hybrid search****

In [100]:
import pandas as pd

ks = [1, 50, 100, 200]
mrrs = {}

for k in ks:
    print(f"Running hybrid_search with k={k}...")
    mrrs[k] = evaluate(ground_truth, lambda query, k=k: hybrid_search(query, k=k))["mrr"]

pd.DataFrame.from_dict(mrrs, orient="index", columns=["mrr"])

Running hybrid_search with k=1...


  0%|          | 0/15 [00:00<?, ?it/s]

Running hybrid_search with k=50...


  0%|          | 0/15 [00:00<?, ?it/s]

Running hybrid_search with k=100...


  0%|          | 0/15 [00:00<?, ?it/s]

Running hybrid_search with k=200...


  0%|          | 0/15 [00:00<?, ?it/s]

,mrr
1,0.627778
50,0.583333
100,0.583333
200,0.583333
